# Day-wise Trajectory Preprocessing and Checkpointing
This cell automates preprocessing for all available days using 15-minute interpolation, storing vessel trajectories as checkpoints for fast future loading.

In [2]:
import os
import pickle
import pandas as pd
from pathlib import Path
from scipy.interpolate import CubicSpline
from datetime import timedelta
import numpy as np
import warnings
from math import radians, cos, sin, asin, sqrt

warnings.filterwarnings("ignore")

# Directories
data_dir = Path("../processed_data")
output_dir = Path("../outputs_preprocessing")
output_dir.mkdir(exist_ok=True)

# Find all daily pickle files
pkl_files = sorted(data_dir.glob("AIS_*.pkl"))
print(f"Found {len(pkl_files)} daily pickle files.")

# ==================== MARITIME CALCULATIONS ====================
# Functions from 03_loitering._parameterization.ipynb


def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance in nautical miles between two points."""
    R = 3440.065  # Earth's radius in nautical miles
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return R * c


def calculate_course(lat1, lon1, lat2, lon2):
    """Calculate Course Over Ground (COG) in degrees (0-360)."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(dlon)
    cog = (np.arctan2(x, y) * 180 / np.pi) % 360
    return cog


def calculate_course_change(cog_curr, cog_prev):
    """Calculate course change (normalized to [0, 180] degrees)."""
    if cog_prev is None or np.isnan(cog_prev) or np.isnan(cog_curr):
        return 0.0
    delta_c = abs(cog_curr - cog_prev)
    if delta_c > 180:
        return 360 - delta_c
    return delta_c


def calculate_speed(lat1, lon1, lat2, lon2, time_delta_seconds):
    """Calculate speed in knots."""
    if time_delta_seconds <= 0:
        return 0.0
    distance_nm = haversine(lat1, lon1, lat2, lon2)
    time_hours = time_delta_seconds / 3600.0
    speed = distance_nm / time_hours if time_hours > 0 else 0
    return speed


def method1_trajectory_redundancy(df):
    """
    METHOD 1: Basic Trajectory Redundancy (Equation 1)
    ψ = D / P where D = total dynamic trajectory length, P = perimeter of bounding box
    """
    if len(df) < 2:
        return 0.0, 0.0, 0.0
    dists = []
    for i in range(len(df) - 1):
        d = haversine(
            df["lat"].iloc[i],
            df["lon"].iloc[i],
            df["lat"].iloc[i + 1],
            df["lon"].iloc[i + 1],
        )
        dists.append(d)
    D = np.sum(dists)
    min_lat, max_lat = df["lat"].min(), df["lat"].max()
    min_lon, max_lon = df["lon"].min(), df["lon"].max()
    height = haversine(min_lat, min_lon, max_lat, min_lon)
    width = haversine(min_lat, min_lon, min_lat, max_lon)
    P = 2 * (height + width)
    if P == 0:
        return 0.0, D, P
    psi = D / P
    return psi, D, P


def method2_loitering_score_Fc(df):
    """
    METHOD 2: Loitering Score F(c) (Equation 6)
    F(c) = (Σ Ck × Σ Sk) / (180 × B)
    """
    if len(df) < 2:
        return 0.0, 0.0, 0.0
    df = df.sort_values("timestamp").reset_index(drop=True)
    course_changes = []
    speeds = []
    for i in range(1, len(df)):
        lat_prev, lon_prev = df["lat"].iloc[i - 1], df["lon"].iloc[i - 1]
        lat_curr, lon_curr = df["lat"].iloc[i], df["lon"].iloc[i]
        cog = calculate_course(lat_prev, lon_prev, lat_curr, lon_curr)
        cog_prev = calculate_course(
            df["lat"].iloc[i - 2] if i > 1 else lat_prev,
            df["lon"].iloc[i - 2] if i > 1 else lon_prev,
            lat_prev,
            lon_prev,
        )
        delta_c = calculate_course_change(cog, cog_prev)
        course_changes.append(delta_c)
        time_delta = (
            df["timestamp"].iloc[i] - df["timestamp"].iloc[i - 1]
        ).total_seconds()
        spd = calculate_speed(lat_prev, lon_prev, lat_curr, lon_curr, time_delta)
        speeds.append(spd)
    sum_course_changes = np.sum(course_changes)
    sum_speeds = np.sum(speeds)
    min_lat, max_lat = df["lat"].min(), df["lat"].max()
    min_lon, max_lon = df["lon"].min(), df["lon"].max()
    height = haversine(min_lat, min_lon, max_lat, min_lon)
    width = haversine(min_lat, min_lon, min_lat, max_lon)
    B = height * width if height > 0 and width > 0 else 1.0
    Fc = (sum_course_changes * sum_speeds) / (180 * B) if B > 0 else 0.0
    return Fc, sum_course_changes, sum_speeds


# ==================== TRAJECTORY INTERPOLATION ====================
# Functions from 02_trajectory_parametrization.ipynb


def interpolate_vessel_trajectory(
    df_vessel,
    target_interval_minutes=15,
    time_col="BaseDateTime",
    lat_col="Latitude",
    lon_col="Longitude",
):
    """Interpolate vessel trajectory at regular intervals using cubic spline."""
    if len(df_vessel) < 2:
        return pd.DataFrame()
    required_cols = [time_col, lat_col, lon_col]
    if not all(col in df_vessel.columns for col in required_cols):
        return pd.DataFrame()
    df = df_vessel[[time_col, lat_col, lon_col]].dropna().copy()
    df = df.sort_values(time_col).reset_index(drop=True)
    if len(df) < 2:
        return pd.DataFrame()
    t_start = df[time_col].min()
    t_numeric = (df[time_col] - t_start).dt.total_seconds().values
    lat_vals = df[lat_col].values
    lon_vals = df[lon_col].values
    try:
        cs_lat = CubicSpline(t_numeric, lat_vals, bc_type="not-a-knot")
        cs_lon = CubicSpline(t_numeric, lon_vals, bc_type="not-a-knot")
    except Exception as e:
        print(f"  Warning: spline fit failed - {e}")
        return pd.DataFrame()
    t_min, t_max = t_numeric.min(), t_numeric.max()
    dt_seconds = target_interval_minutes * 60
    t_interp = np.arange(t_min, t_max + dt_seconds, dt_seconds)
    lat_interp = cs_lat(t_interp)
    lon_interp = cs_lon(t_interp)
    timestamps_interp = [t_start + timedelta(seconds=float(t)) for t in t_interp]
    is_interpolated = ~np.isin(t_interp, t_numeric)
    result = pd.DataFrame(
        {
            "timestamp": timestamps_interp,
            "lat": lat_interp,
            "lon": lon_interp,
            "interpolated": is_interpolated,
        }
    )
    return result


def preprocess_day_data(
    df_day, date_str, output_dir=output_dir, target_interval_minutes=15
):
    """
    Complete preprocessing pipeline for a single day:
    1. Load raw vessel data
    2. Interpolate trajectories at 15-minute intervals
    3. Calculate trajectory redundancy metrics
    4. Store checkpoint with all processed data
    """
    time_col = "BaseDateTime" if "BaseDateTime" in df_day.columns else "timestamp"
    lat_col = "Latitude" if "Latitude" in df_day.columns else "lat"
    lon_col = "Longitude" if "Longitude" in df_day.columns else "lon"
    mmsi_col = "MMSI" if "MMSI" in df_day.columns else "mmsi"
    required_cols = [mmsi_col, time_col, lat_col, lon_col]
    if not all(col in df_day.columns for col in required_cols):
        print(f"❌ Missing required columns. Available: {df_day.columns.tolist()}")
        return {}

    df_subset = df_day[required_cols].copy()
    processed_vessels = {}

    print(f"  Processing {len(df_subset[mmsi_col].unique())} vessels...")

    for mmsi, group in df_subset.groupby(mmsi_col):
        # Step 1: Interpolate trajectory
        interp_traj = interpolate_vessel_trajectory(
            group, target_interval_minutes, time_col, lat_col, lon_col
        )
        if len(interp_traj) > 0:
            # Step 2: Calculate trajectory metrics
            redundancy_psi, path_length, bbox_perim = method1_trajectory_redundancy(
                interp_traj
            )
            loitering_Fc, sum_courses, sum_speeds = method2_loitering_score_Fc(
                interp_traj
            )

            # Store with metadata
            processed_vessels[mmsi] = {
                "trajectory": interp_traj,
                "metrics": {
                    "redundancy_psi": redundancy_psi,
                    "path_length_nm": path_length,
                    "bbox_perimeter_nm": bbox_perim,
                    "loitering_score_Fc": loitering_Fc,
                    "total_course_changes": sum_courses,
                    "total_distance_nm": sum_speeds,
                    "num_points": len(interp_traj),
                    "interpolation_interval_min": target_interval_minutes,
                },
            }

    # Step 3: Save checkpoint
    checkpoint_file = output_dir / f"interpolated_{date_str}.pkl"
    with open(checkpoint_file, "wb") as f:
        pickle.dump(processed_vessels, f)
    print(
        f"  📁 Checkpoint saved: {checkpoint_file} ({len(processed_vessels)} vessels)"
    )
    return processed_vessels


# Main loop: preprocess all days
print("\n" + "=" * 80)
print("STARTING DAY-WISE PREPROCESSING WITH 15-MINUTE INTERPOLATION")
print("=" * 80 + "\n")

successful_days = 0
failed_days = 0

for pkl_file in pkl_files:
    date_str = pkl_file.stem.replace("AIS_", "")
    print(f"\nProcessing {date_str}...")
    try:
        with open(pkl_file, "rb") as f:
            day_data = pickle.load(f)
        if isinstance(day_data, pd.DataFrame):
            preprocess_day_data(day_data, date_str)
            successful_days += 1
        elif isinstance(day_data, dict):
            # If dict, try to concatenate all vessel data
            all_df = pd.concat(day_data.values(), ignore_index=True)
            preprocess_day_data(all_df, date_str)
            successful_days += 1
        else:
            print(f"Unknown data type for {date_str}: {type(day_data)}")
            failed_days += 1
    except Exception as e:
        print(f"✗ Failed to process {date_str}: {e}")
        failed_days += 1

print("\n" + "=" * 80)
print(f"PREPROCESSING COMPLETE")
print(f"  ✓ Successful: {successful_days} days")
print(f"  ✗ Failed: {failed_days} days")
print("=" * 80)

Found 6 daily pickle files.

STARTING DAY-WISE PREPROCESSING WITH 15-MINUTE INTERPOLATION


Processing 2020_01_03...
❌ Missing required columns. Available: ['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'VesselName', 'CallSign', 'VesselType', 'Status', 'Length', 'Width', 'Draft', 'Cargo', 'TransceiverClass', 'hour', 'dayofweek', 'month', 'is_weekend', 'v_x', 'v_y', 'turn_rate', 'accel_knots_per_hr', 'is_stopped', 'Δt_hours', 'ΔCOG', 'ΔSOG', 'COG_rad', 'IMO_IMO0000000', 'IMO_IMO0000001', 'IMO_IMO0000019', 'IMO_IMO0000032', 'IMO_IMO0000111', 'IMO_IMO0000168', 'IMO_IMO0001242', 'IMO_IMO0507027', 'IMO_IMO0600071', 'IMO_IMO0606904', 'IMO_IMO0801801', 'IMO_IMO0972768', 'IMO_IMO1002055', 'IMO_IMO1002093', 'IMO_IMO10024346', 'IMO_IMO1002990', 'IMO_IMO1003621', 'IMO_IMO1003970', 'IMO_IMO100538269', 'IMO_IMO1006269', 'IMO_IMO100633661', 'IMO_IMO1006544', 'IMO_IMO1006738', 'IMO_IMO1006881', 'IMO_IMO1006910', 'IMO_IMO1007237', 'IMO_IMO1007328', 'IMO_IMO1007419', 'IMO_IMO1007445', 